## Sakila Data Lakehouse with PySpark Structured Streaming
This notebook demonstrates a streaming ETL pipeline using the Sakila database, implementing the Medallion Architecture (Bronze, Silver, Gold layers).

### Business Process: DVD Rental Transactions
The data lakehouse models rental transactions, payments, and inventory movements.

### Prerequisites:
- `python -m pip install pymongo[srv]`
- `python -m pip install pymysql`
- `python -m pip install sqlalchemy`
- `python -m pip install pyspark`

## Section I: Prerequisites
### 1.0. Import Required Libraries

In [1]:
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark


### 2.0. Instantiate Global Variables

In [ ]:
# MySQL connection for source Sakila database
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "sakila_dw",
    "conn_props" : {
        "user" : "root",
        "password" : "",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}

# MongoDB connection
mongodb_args = {
    "cluster_location" : "atlas",
    "user_name" : "nathanluu",
    "password" : "dFq6YFa5c6ggnmMf",
    "cluster_name" : "cluster0",
    "cluster_subnet" : "7ycvikt",
    "db_name" : "sakila",
    "collection" : "",
    "null_column_threshold" : 0.5
}

# Directory structure
base_dir = os.path.join(os.getcwd(), 'data')
data_dir = os.path.join(base_dir, 'sakila')
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'stream')

rentals_stream_dir = os.path.join(stream_dir, 'rentals')
payments_stream_dir = os.path.join(stream_dir, 'payments')
inventory_stream_dir = os.path.join(stream_dir, 'inventory_transactions')

# Data Lakehouse structure
dest_database = "sakila_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

rentals_output_bronze = os.path.join(database_dir, 'fact_rentals', 'bronze')
rentals_output_silver = os.path.join(database_dir, 'fact_rentals', 'silver')
rentals_output_gold = os.path.join(database_dir, 'fact_rentals', 'gold')

payments_output_bronze = os.path.join(database_dir, 'fact_payments', 'bronze')
payments_output_silver = os.path.join(database_dir, 'fact_payments', 'silver')
payments_output_gold = os.path.join(database_dir, 'fact_payments', 'gold')

### 3.0. Define Global Functions

In [3]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])
    
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))
    
    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    return pd.DataFrame(data=data, columns=column_names)

def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
    print(f"The stream has processed {len(query.recentProgress)} batches")

def remove_directory_tree(path: str):
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
    except Exception as e:
        return f"An error occurred: {e}"

def get_mysql_dataframe(spark_session, sql_query: str, **args):
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    dframe = spark_session.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("driver", args['conn_props']['driver']) \
        .option("user", args['conn_props']['user']) \
        .option("password", args['conn_props']['password']) \
        .option("query", sql_query) \
        .load()
    return dframe

def get_mongo_uri(**args):
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
    
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"
    return uri

def get_spark_conf_args(spark_jars: list, **args):
    jars = ""
    for jar in spark_jars:
        jars += f"{jar}, "
    
    sparkConf_args = {
        "app_name" : "Capstone",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars[:-1],
        "database_dir" : sql_warehouse_dir
    }
    return sparkConf_args

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name'])\
        .setMaster(args['worker_threads']) \
        .set('spark.driver.memory', '4g') \
        .set('spark.executor.memory', '2g') \
        .set('spark.jars', args['spark_jars']) \
        .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:3.0.1') \
        .set('spark.mongodb.input.uri', args['mongo_uri']) \
        .set('spark.mongodb.output.uri', args['mongo_uri']) \
        .set('spark.sql.adaptive.enabled', 'false') \
        .set('spark.sql.debug.maxToStringFields', 35) \
        .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
        .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
        .set('spark.sql.streaming.schemaInference', 'true') \
        .set('spark.sql.warehouse.dir', args['database_dir']) \
        .set('spark.streaming.stopGracefullyOnShutdown', 'true')
    return sparkConf

### 4.0. Initialize Data Lakehouse Directory Structure

In [4]:
remove_directory_tree(database_dir)

"Directory '/Users/nathancluu/DS-2002/Projects/Project 2/spark-warehouse/sakila_dlh.db' has been removed successfully."

### 5.0. Create a New Spark Session

In [5]:
jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")
jars.append(mysql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)
sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark

25/12/17 17:05:39 WARN Utils: Your hostname, Nathans-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.86.54 instead (on interface en0)
25/12/17 17:05:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/nathancluu/.ivy2/cache
The jars for the packages stored in: /Users/nathancluu/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3eaf27f1-66eb-41fc-b6e6-79453cb2b36a;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 in central
	found org.mongodb#mongodb-driver-sync;4.0.5 in central
	found org.mongodb#bson;4.0.5 in central
	found org.mongodb#mongodb-driver-core;4.0.5 in central
:: resolution report :: resolve 75ms :: artifacts dl 4ms
	:: modules in use:
	org.mongodb#bson;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-core;4.0.5 from central in [default]
	org.mongodb#mongodb-dr

:: loading settings :: url = jar:file:/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


25/12/17 17:05:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### 6.0. Create a New Metadata Database

In [6]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'Sakila Streaming Data Lakehouse'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'Sakila Streaming ETL');
"""
spark.sql(sql_create_db)

DataFrame[]

## Section II: Populate Dimensions from Cold-Path Data
### 1.0. Populate dim_date from MySQL

All dimensional tables in MySQL were originally created from work done in Project 1. See Project 1 notebook to see that data was originated from multiple sources (MySQL, MongoDB Atlas, and local file system).

In [7]:
# Extract date dimension
sql_dim_date = f"SELECT * FROM {mysql_args['db_name']}.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)

# Load to lakehouse
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

# Verify
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 2").toPandas()

,DateKey,DateValue,DayOfMonth,Month,MonthName,Quarter,Year,WeekOfYear,DayOfWeek,DayName,IsWeekend
0,20050101,2005-01-01,1,1,January,1,2005,0,7,Saturday,1
1,20050102,2005-01-02,2,1,January,1,2005,0,1,Sunday,1


### 2.0. Populate dim_customers from MySQL

In [8]:
sql_customers = """
    SELECT 
        c.customer_id,
        c.first_name,
        c.last_name,
        c.email,
        c.active,
        ci.city,
        co.country
    FROM sakila.customer c
    INNER JOIN sakila.address a ON c.address_id = a.address_id
    INNER JOIN sakila.city ci ON a.city_id = ci.city_id
    INNER JOIN sakila.country co ON ci.country_id = co.country_id
"""

df_dim_customers = get_mysql_dataframe(spark, sql_customers, **mysql_args)

# Add surrogate key
df_dim_customers.createOrReplaceTempView("customers")
sql_customers_pk = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key
    FROM customers
"""
df_dim_customers = spark.sql(sql_customers_pk)

# Reorder columns
ordered_columns = ['customer_key', 'customer_id', 'first_name', 'last_name', 
                   'email', 'active', 'city', 'country']
df_dim_customers = df_dim_customers[ordered_columns]

# Load to lakehouse
df_dim_customers.write.saveAsTable(f"{dest_database}.dim_customers", mode="overwrite")

# Verify
spark.sql(f"SELECT * FROM {dest_database}.dim_customers LIMIT 2").toPandas()

,customer_key,customer_id,first_name,last_name,email,active,city,country
0,1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,True,Sasebo,Japan
1,2,2,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,True,San Bernardino,United States


### 3.0. Populate dim_films from MySQL

In [9]:
sql_films = """
    SELECT 
        f.film_id,
        f.title,
        f.release_year,
        f.rental_duration,
        f.rental_rate,
        f.length,
        f.replacement_cost,
        f.rating,
        c.name AS category
    FROM sakila.film f
    LEFT JOIN sakila.film_category fc ON f.film_id = fc.film_id
    LEFT JOIN sakila.category c ON fc.category_id = c.category_id
"""

df_dim_films = get_mysql_dataframe(spark, sql_films, **mysql_args)

# Add surrogate key
df_dim_films.createOrReplaceTempView("films")
sql_films_pk = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY film_id) AS film_key
    FROM films
"""
df_dim_films = spark.sql(sql_films_pk)

# Reorder columns
ordered_columns = ['film_key', 'film_id', 'title', 'category', 'release_year',
                   'rental_duration', 'rental_rate', 'length', 'replacement_cost', 'rating']
df_dim_films = df_dim_films[ordered_columns]

# Load to lakehouse
df_dim_films.write.saveAsTable(f"{dest_database}.dim_films", mode="overwrite")

# Verify
spark.sql(f"SELECT * FROM {dest_database}.dim_films LIMIT 2").toPandas()

,film_key,film_id,title,category,release_year,rental_duration,rental_rate,length,replacement_cost,rating
0,1,1,ACADEMY DINOSAUR,Documentary,2006-01-01,6,0.99,86,20.99,PG
1,2,2,ACE GOLDFINGER,Horror,2006-01-01,3,4.99,48,12.99,G


### 4.0. Populate dim_staff from MySQL

In [10]:
sql_staff = """
    SELECT 
        s.staff_id,
        s.first_name,
        s.last_name,
        s.email,
        s.active,
        s.store_id
    FROM sakila.staff s
"""

df_dim_staff = get_mysql_dataframe(spark, sql_staff, **mysql_args)

# Add surrogate key
df_dim_staff.createOrReplaceTempView("staff")
sql_staff_pk = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY staff_id) AS staff_key
    FROM staff
"""
df_dim_staff = spark.sql(sql_staff_pk)

# Reorder columns
ordered_columns = ['staff_key', 'staff_id', 'first_name', 'last_name', 
                   'email', 'active', 'store_id']
df_dim_staff = df_dim_staff[ordered_columns]

# Load to lakehouse
df_dim_staff.write.saveAsTable(f"{dest_database}.dim_staff", mode="overwrite")

# Verify
spark.sql(f"SELECT * FROM {dest_database}.dim_staff LIMIT 2").toPandas()

,staff_key,staff_id,first_name,last_name,email,active,store_id
0,1,1,Mike,Hillyer,Mike.Hillyer@sakilastaff.com,True,1
1,2,2,Jon,Stephens,Jon.Stephens@sakilastaff.com,True,2


### 5.0. Populate dim_stores from MySQL

In [11]:
sql_stores = "SELECT * FROM sakila.store"
df_dim_stores = get_mysql_dataframe(spark, sql_stores, **mysql_args)

# Add surrogate key
df_dim_stores.createOrReplaceTempView("stores")
sql_stores_pk = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY store_id) AS store_key
    FROM stores
"""
df_dim_stores = spark.sql(sql_stores_pk)

# Reorder columns
ordered_columns = ['store_key', 'store_id', 'manager_staff_id', 'address_id']
df_dim_stores = df_dim_stores[ordered_columns]

# Load to lakehouse
df_dim_stores.write.saveAsTable(f"{dest_database}.dim_stores", mode="overwrite")

# Verify
spark.sql(f"SELECT * FROM {dest_database}.dim_stores LIMIT 2").toPandas()

,store_key,store_id,manager_staff_id,address_id
0,1,1,1,1
1,2,2,2,2


### 6.0. Verify All Dimension Tables

In [12]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,sakila_dlh,dim_customers,False
1,sakila_dlh,dim_date,False
2,sakila_dlh,dim_films,False
3,sakila_dlh,dim_staff,False
4,sakila_dlh,dim_stores,False
5,,customers,True
6,,films,True
7,,staff,True
8,,stores,True


## Section III: Process Hot-Path Streaming Data
### 1.0. Process Rentals Fact Data
#### 1.1. Verify Source Data Files

Source Data Files come from MySQL fact rental table that was split into 3 chunks.

In [13]:
get_file_info(rentals_stream_dir)

,name,size,modification_time
0,.DS_Store,6148,2025-12-16 21:39:02.820455551
1,rentals01.json,83586,2025-12-16 21:35:00.667911768
2,rentals02.json,83718,2025-12-16 21:35:16.220991373
3,rentals03.json,84175,2025-12-16 21:35:36.641285181


#### 1.2. Bronze Layer: Stage Raw Rental Data

In [14]:
# Read streaming data
df_rentals_bronze = (
    spark.readStream \
    .option("schemaLocation", rentals_output_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiLine", "true") \
    .json(rentals_stream_dir)
)

# Write to silver layer
rentals_checkpoint_bronze = os.path.join(rentals_output_bronze, '_checkpoint')

rentals_bronze_query = (
    df_rentals_bronze
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("rentals_bronze")
    .trigger(availableNow = True) \
    .option("checkpointLocation", rentals_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(rentals_output_bronze)
)

print(f"Query Name: {rentals_bronze_query.name}")
rentals_bronze_query.awaitTermination()

df_bronze_static = spark.read.parquet(rentals_output_bronze)
df_bronze_static.show(2, truncate=False)


Query Name: rentals_bronze
+------+------------+--------+----------------+----------+---------------+---------+----------+---------------+---------+-----------------------+-----------------------------------------------------------------------------------------------+
|amount|customer_key|film_key|payment_date_key|payment_id|rental_date_key|rental_id|rental_key|return_date_key|store_key|receipt_time           |source_file                                                                                    |
+------+------------+--------+----------------+----------+---------------+---------+----------+---------------+---------+-----------------------+-----------------------------------------------------------------------------------------------+
|0.99  |35          |1       |20050708        |11630     |20050708       |4863     |1         |20050711       |1        |2025-12-17 17:05:44.501|file:///Users/nathancluu/DS-2002/Projects/Project%202/data/sakila/stream/rentals/rentals01.json|
|3.99

#### 1.3. Silver Layer: Join with Dimensions

In [15]:
# Prepare role-playing dimensions
df_dim_rental_date = df_dim_date.select(
    col("DateKey").alias("rental_date_key"), 
    col("DateValue").alias("rental_full_date")
)

df_dim_return_date = df_dim_date.select(
    col("DateKey").alias("return_date_key"), 
    col("DateValue").alias("return_full_date")
)

# Read bronze and join with dimensions
df_rentals_silver = spark.readStream.format("parquet").load(rentals_output_bronze) \
    .join(df_dim_customers, "customer_key", "inner") \
    .join(df_dim_films, "film_key", "inner") \
    .join(df_dim_stores, "store_key", "inner") \
    .select(
        col("rental_key").cast(LongType()),
        col("customer_key").cast(LongType()),
        col("film_key").cast(LongType()),
        col("store_key").cast(LongType()),
        col("rental_date_key").cast(LongType()),
        col("return_date_key").cast(LongType())
    )

# Write to silver layer
rentals_checkpoint_silver = os.path.join(rentals_output_silver, '_checkpoint')

rentals_silver_query = (
    df_rentals_silver.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("rentals_silver")
    .trigger(availableNow = True) \
    .option("checkpointLocation", rentals_checkpoint_silver) \
    .option("compression", "snappy") \
    .start(rentals_output_silver)
)

print(f"Query Name: {rentals_silver_query.name}")
rentals_silver_query.awaitTermination()

df_silver_static = spark.read.parquet(rentals_output_silver)
df_silver_static.show(2, truncate=False)

Query Name: rentals_silver
+----------+------------+--------+---------+---------------+---------------+
|rental_key|customer_key|film_key|store_key|rental_date_key|return_date_key|
+----------+------------+--------+---------+---------------+---------------+
|243       |44          |28      |1        |20050712       |20050721       |
|238       |83          |28      |1        |20050708       |20050711       |
+----------+------------+--------+---------+---------------+---------------+
only showing top 2 rows



#### 1.4. Gold Layer: Rentals by Film Category per Month

This query provides insight into which film categories were most popular in each month, helping the business understand trends in customer rentals.

In [16]:
df_rentals_by_category_gold = spark.readStream.format("parquet").load(rentals_output_silver) \
    .join(df_dim_films, "film_key") \
    .join(df_dim_date, col("rental_date_key") == col("DateKey")) \
    .groupBy(
        col("Year"),
        col("Month").alias("month_of_year"),
        col("MonthName").alias("month_name"),
        col("category")
    ) \
    .agg(count("rental_key").alias("total_rentals"))

rentals_gold_query = (
    df_rentals_by_category_gold.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("fact_rentals_by_category") \
    .start()
)

wait_until_stream_is_ready(rentals_gold_query, 1)

df_fact_rentals_by_category_final = spark.sql("SELECT * FROM fact_rentals_by_category") \
    .select(
        col("Year"),
        col("month_name").alias("Month"),
        col("category").alias("Film Category"),
        col("total_rentals").alias("Total Rentals"),
    )

df_fact_rentals_by_category_final.write.saveAsTable(
    f"{dest_database}.fact_rentals_by_category",
    mode="overwrite"
)


spark.sql(
    f"""
    SELECT *
    FROM {dest_database}.fact_rentals_by_category
    ORDER BY `Total Rentals` DESC
    """
).toPandas()

The stream has processed 1 batches


,Year,Month,Film Category,Total Rentals
0,2005,July,Sci-Fi,44
1,2005,July,New,43
2,2005,August,Sci-Fi,38
3,2005,July,Family,38
4,2005,July,Horror,37
...,...,...,...,...
66,2006,February,Sci-Fi,1
67,2005,May,Sports,1
68,2005,May,Classics,1
69,2006,February,Animation,1


In [17]:
spark.stop()